# Normalization

## 1. LayerNorm

Given a hidden state $a$ from a Transformer, LayerNorm first computes its mean $\mu$ and standard variance $\sigma$, and then performs e-centering and re-scaling. The procedure is as follows:

$\bar{a}=\frac{a-\mu}\sigma*g_i$

where $g_i$ is a learnable scaling vector.



In [ ]:
import torch
import torch.nn as nn
class LayerNorm(nn.Module):

    def __init__(self,hidden_size,eps=1e-5):
        super().__init__()
        self.hidden_size=hidden_size
        self.eps=eps
        self.scaling=nn.Parameter(torch.ones(hidden_size))
        self.bias=nn.Parameter(torch.ones(hidden_size))
    
    def forward(self,x):
        mu=torch.mean(x,dim=-1,keepdim=True)
        sigma=torch.std(x,dim=-1,keepdim=True, unbiased=False)
        x=(x-mu)/(sigma+self.eps)
        return x*self.scaling

LN=LayerNorm(512)
input=torch.randn(2,6,512)
print(input)
print(LN(input))


## 2. RMSNorm
Different from LayerNorm, RMSNorm can be depicted as:

$\bar{a}=\frac{a}{rms}*g_i$

where $rms$ denotes root mean square.

In [ ]:
class RMSNorm(nn.Module):

    def __init__(self,hidden_size,eps=0):
        super().__init__()
        self.hidden_size=hidden_size
        self.eps=eps
        self.scaling=nn.Parameter(torch.ones(hidden_size))

    def forward(self,x):
        RMS=x.pow(2).mean(dim=-1, keepdim=True)
        x=x*torch.sqrt(self.eps+RMS)
        return x*self.scaling

input=torch.randn(2,3,512)
my_RMSNorm=RMSNorm(512)
print(my_RMSNorm(input))


# Qwen3ASRTextAttention

In [ ]:
import torch
import torch.nn as nn

class Qwen3ASRTextAttention(nn.Module):
    
    # We leave the implementation of Qwen3ASRTextRMSNorm, pos_emb, mask, dropout, and kv Cache in the futrue

    def __init__(self,hidden_size,num_q_head,num_kv_head):

        self.hidden_size=hidden_size
        self.num_q_head=num_q_head
        self.num_kv_head=num_kv_head
        self.head_dim=hidden_size//num_q_head
        self.kv_group=num_q_head//num_kv_head
        self.scaling=self.head_dim**-0.5

        self.q=nn.Linear(hidden_size,self.head_dim*num_q_head)
        self.k=nn.Linear(hidden_size,self.head_dim*num_kv_head)
        self.v=nn.Linear(hidden_size,self.head_dim*num_kv_head)
        self.o=nn.Linear(hidden_size,hidden_size)
		
    def forward(self,x):
        
    	# x [B,T,F]
        B,T,_=x.shape()
        
    	# forward QKV
        # TODO why T should be at the dim=2 rather than dim=1 ?
        Q=self.q(x).view(B,self.num_q_head,T,self.head_dim)
        K=self.k(x).view(B,self.num_kv_head,T,self.head_dim)
        V=self.v(x).view(B,self.num_kv_head,T,self.head_dim)
        
        # repeat K and V
        K=torch.repeat_interleave(K,dim=1,repeats=self.kv_group)
        V=torch.repeat_interleave(V,dim=1,repeats=self.kv_group)
        
        # comute attention scores
        attn_weights=Q@K.transpose(-2,-1)*self.scaling
        attn_weights=nn.functional.softmax(attn_weights,dim=-1)
        output=(attn_weights@V).view(B,T,-1)
        output=self.o(output)

        return output, attn_weights